# purple-sectors · 01 · Recovery, then real 2026 fit
**Part 1:** simulate a season from known parameters on the real design, fit Model A (main) and Model B (neutral car-share prior), check both recover the truth.
**Part 2:** only if Part 1 passes, fit both on the real 2026 data and export the posterior for analysis.

Runtime ≈ 20 min (CmdStan install 5 + four fits).

In [ ]:
# 1 · Drag the newest purple-sectors zip into the Files panel first, then run
import glob, io, os, zipfile
from google.colab import files
os.chdir('/content')
z = sorted(glob.glob('/content/purple-sectors*.zip'), key=os.path.getmtime)
assert z, 'No purple-sectors zip in /content. Drag it into the Files panel and wait for the upload.'
zipfile.ZipFile(z[-1]).extractall('/content')
os.chdir('/content/purple-sectors')
print('using', z[-1])

In [ ]:
# 2 · Install CmdStan (~5 min, once per Colab session)
%pip install -q cmdstanpy plotly
import cmdstanpy
cmdstanpy.install_cmdstan(progress=True)

In [ ]:
# 3 · Load the real design (one car per driver; substitutes excluded)
import os, sys
os.chdir('/content/purple-sectors')        # survives a Colab session restart
sys.path.insert(0, '/content/purple-sectors/src')
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cmdstanpy import CmdStanModel
from purple_sectors.prepare import load_push_laps, build_design
from purple_sectors.simulate import draw_params, simulate_laps

laps = load_push_laps('data/raw/quali_laps_2026.csv')
des = build_design(laps)
print(f'{des.N} push laps · {des.D} drivers · {des.C} cars · {des.E} events')

In [ ]:
# 4 · Helpers: fitting, diagnostics, styling
MODELS = {'A': CmdStanModel(stan_file='stan/quali.stan'),
          'B': CmdStanModel(stan_file='stan/quali_share.stan')}
HYPER = ['m', 'sigma_kappa', 'sigma_car', 'sigma_dev', 'sigma_nu', 'sigma_driver',
         'beta_m', 'beta_a', 'sigma', 'loss']

def fit_model(key, y, seed=1):
    return MODELS[key].sample(data=des.stan_data(y), chains=4, parallel_chains=4,
                              iter_warmup=1500, iter_sampling=3000, adapt_delta=0.95,
                              inits=0.5, seed=seed, show_progress=False)

def max_rhat(fit, vars):
    df = fit.draws_pd(vars=vars)
    cols = [c for c in df.columns if not c.endswith('__')]
    n_it = fit.num_draws_sampling
    x = df[cols].to_numpy().reshape(fit.chains, n_it, len(cols)).transpose(1, 0, 2)
    h = n_it // 2
    x = np.concatenate([x[:h], x[h:2 * h]], axis=1)              # split chains
    W = x.var(axis=0, ddof=1).mean(axis=0)
    B = h * x.mean(axis=0).var(axis=0, ddof=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        r = np.sqrt(((h - 1) / h * W + B / h) / W)
    return np.nanmax(r)

def health(fit, name):
    div = int(fit.method_variables()['divergent__'].sum())
    rh = max_rhat(fit, HYPER + ['delta', 'theta', 'nu', 'tau'])
    flag = 'OK' if div == 0 and rh < 1.01 else 'CHECK'
    print(f'{name:<22} divergences: {div:<4} max R-hat: {rh:.3f}   {flag}')

INK, MUTED, GRID = '#2c2c2a', '#898781', '#ecebe6'
BLUE, BLUE_LT, ORANGE = '#2a78d6', '#b5d4f4', '#eb6834'

def style(fig, title, subtitle='', h=460):
    fig.update_layout(
        template='plotly_white', height=h, showlegend=False,
        title=dict(text=f'<b>{title}</b><br><span style="font-size:12px;color:{MUTED}">{subtitle}</span>', x=0),
        font=dict(family='Inter, system-ui, sans-serif', size=12, color=INK),
        margin=dict(l=10, r=20, t=80, b=40), hoverlabel=dict(font_size=12))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID)
    return fig

def interval_plot(draws, true, labels, title, subtitle, h=None):
    lo, med, hi = np.percentile(draws, [5, 50, 95], axis=0)
    order = np.argsort(med)
    inside = (true >= lo) & (true <= hi)
    lab = np.array(labels)[order]
    fig = go.Figure()
    for j in order:
        fig.add_trace(go.Scatter(x=[lo[j], hi[j]], y=[labels[j]] * 2, mode='lines',
                                 line=dict(color=BLUE_LT, width=8), hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=med[order], y=lab, mode='markers', marker=dict(color=BLUE, size=9)))
    fig.add_trace(go.Scatter(x=true[order], y=lab, mode='markers',
                             marker=dict(color=ORANGE, size=11, symbol='x-thin', line=dict(width=2.5, color=ORANGE))))
    return style(fig, title, f'{subtitle} · truth inside 90% interval: {inside.mean():.0%}',
                 h or 28 * len(labels) + 140)

def cover(draws, true):
    lo, hi = np.percentile(draws, [5, 95], axis=0)
    return ((true >= lo) & (true <= hi)).mean()

def car_share_true(p):
    vc = (p['theta'] + p['nu']).var(axis=1, ddof=1)
    return vc / (vc + p['delta'].var(ddof=1))

## Part 1 · Parameter recovery

In [ ]:
# 5 · Simulate a fake season from known, realistic parameters, then fit A and B
rng = np.random.default_rng(42)
TRUTH = dict(m=np.array([-0.55, -0.40]), sigma_kappa=0.15, sigma_car=1.5, sigma_dev=0.25,
             sigma_nu=0.30, sigma_driver=0.30, beta_m=0.8, beta_a=0.04, sigma=0.25, loss=0.35)
truth = draw_params(des, rng, **TRUTH)
y_sim = simulate_laps(des, truth, rng)

sim = {k: fit_model(k, y_sim) for k in MODELS}
for k, f in sim.items():
    health(f, f'simulated · model {k}')

In [ ]:
# 6 · Hyperparameters (model A): posterior / truth, 1.0 = perfect
names = ['m[1]', 'm[2]'] + HYPER[1:]
true_vals = np.r_[TRUTH['m'], [TRUTH[k] for k in HYPER[1:]]]
draws = sim['A'].draws_pd(vars=HYPER)[names].to_numpy()
fig = interval_plot(draws / true_vals, np.ones(len(names)), names,
                    'Hyperparameter recovery · model A', 'Posterior ÷ truth')
fig.add_vline(x=1, line_color=MUTED, line_dash='dot')
fig.show()

In [ ]:
# 7 · Driver skill (model A)
interval_plot(sim['A'].stan_variable('delta'), truth['delta'], des.drivers,
              'Driver skill recovery · model A',
              '% of lap time, negative = faster · blue = posterior 90%, orange = truth').show()

In [ ]:
# 8 · Car development curves (model A)
theta = sim['A'].stan_variable('theta')
lo, med, hi = np.percentile(theta, [5, 50, 95], axis=0)
x = np.arange(1, des.E + 1)
fig = make_subplots(rows=4, cols=3, subplot_titles=des.cars, shared_xaxes=True,
                    vertical_spacing=0.07, horizontal_spacing=0.06)
for c in range(des.C):
    r, col = divmod(c, 3)
    fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[hi[:, c], lo[::-1, c]], fill='toself',
                             fillcolor='rgba(42,120,214,0.18)', line=dict(width=0), hoverinfo='skip'), r + 1, col + 1)
    fig.add_trace(go.Scatter(x=x, y=med[:, c], line=dict(color=BLUE, width=2)), r + 1, col + 1)
    fig.add_trace(go.Scatter(x=x, y=truth['theta'][:, c], line=dict(color=ORANGE, width=2, dash='dot')), r + 1, col + 1)
style(fig, 'Car pace recovery · model A',
      f'θ by event · band = posterior 90%, dotted = truth · coverage {cover(theta, truth["theta"]):.0%}', h=900)
fig.update_annotations(font_size=12)
fig.show()

In [ ]:
# 9 · Verdict: coverage should be ~90%; A and B should agree on car share
rows = {}
for k, f in sim.items():
    rows[k] = {
        'driver skill δ': cover(f.stan_variable('delta'), truth['delta']),
        'car trend θ': cover(f.stan_variable('theta'), truth['theta']),
        'car × track ν': cover(f.stan_variable('nu'), truth['nu']),
        'track evolution τ': cover(f.stan_variable('tau')[:, :, 1:], truth['tau'][:, 1:]),
        'car share (per event)': cover(f.stan_variable('car_share'), car_share_true(truth)),
    }
print(pd.DataFrame(rows).map('{:.0%}'.format).to_string())

cs = {k: np.percentile(f.stan_variable('car_share').mean(axis=1), [5, 50, 95]) for k, f in sim.items()}
print(f'\nseason car share · truth {car_share_true(truth).mean():.3f}')
for k, (l, m_, h) in cs.items():
    print(f'  model {k}: {m_:.3f}  [{l:.3f}, {h:.3f}]')

## Part 2 · Real 2026 data
Run only if Part 1 shows **0 divergences, R-hat < 1.01, coverage ≈ 80–95%**.

In [ ]:
# 10 · Fit A and B on the real laps
y_real = des.laps['y'].to_numpy()
real = {k: fit_model(k, y_real, seed=2026) for k in MODELS}
for k, f in real.items():
    health(f, f'real 2026 · model {k}')

for k, f in real.items():
    l, m_, h = np.percentile(f.stan_variable('car_share').mean(axis=1), [5, 50, 95])
    print(f'season car share · model {k}: {m_:.3f}  [{l:.3f}, {h:.3f}]')

In [ ]:
# 11 · Export posteriors for analysis (upload the zip back to the chat)
KEEP = ['theta', 'nu', 'delta', 'tau', 'alpha', 'car_share', 'm', 'sigma_kappa', 'sigma_car',
        'sigma_dev', 'sigma_nu', 'sigma_driver', 'beta_m', 'beta_a', 'sigma', 'loss']
meta = dict(drivers=np.array(des.drivers), cars=np.array(des.cars), events=np.array(des.events),
            y=y_real, driver=des.driver, car=des.car, event=des.event, seg=des.seg)
for k, f in real.items():
    out = {v: f.stan_variable(v)[::6].astype('float32') for v in KEEP}    # 2,000 draws
    out['y_rep'] = f.stan_variable('y_rep')[::24].astype('float32')        # 500 draws
    out['log_lik'] = f.stan_variable('log_lik')[::12].astype('float32')    # 1,000 draws, for LOO
    mv = f.method_variables()
    out['divergent'] = mv['divergent__']; out['treedepth'] = mv['treedepth__']
    np.savez_compressed(f'posterior_{k}.npz', **out, **meta)
with zipfile.ZipFile('posteriors_2026.zip', 'w') as z:
    for k in real:
        z.write(f'posterior_{k}.npz')
files.download('posteriors_2026.zip')